# Lesson 3 — Modeling in `discopt`

**Track:** Basic &nbsp;|&nbsp; **Estimated time:** 90 minutes

## Learning objectives

1. Use the `Model` API: variables, constraints, objective.
2. Build expressions with operator overloading and recognize the underlying DAG.
3. Use vectorized variables, indexing, and `sum()` patterns.
4. Detect modeling errors that the solver will turn into infeasibility/unboundedness.

## 1. The four-step recipe

Every `discopt` model is:

```python
m = do.Model("name")
x = m.continuous("x", shape=(n,), lb=0, ub=10)   # 1. variables
m.subject_to(...)                                # 2. constraints
m.minimize(...)                                  # 3. objective
result = m.solve()                               # 4. solve
```

Variable factories: `m.continuous(name, shape=..., lb=..., ub=...)`,
`m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`.
Aggregations like row/column sums use `dm.sum(expr, axis=k)` (not
`expr.sum()`). The expression DAG is built lazily as you compose operators.

See {cite:p}`Williams2013` for the broader modeling philosophy.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
import numpy as np, discopt as do

m = do.Model("transport")
nS, nD = 3, 4
x = m.continuous("x", shape=(nS, nD), lb=0)         # ship from supply i to demand j

supply = np.array([100, 150, 80])
demand = np.array([60, 90, 70, 110])
cost   = np.array([
    [4, 6, 9, 8],
    [5, 4, 7, 6],
    [9, 8, 3, 4],
])

# Supply: row sums <= supply
for i in range(nS):
    m.subject_to(dm.sum(x[i, :]) <= supply[i])

# Demand: column sums >= demand
for j in range(nD):
    m.subject_to(dm.sum(x[:, j]) >= demand[j])

m.minimize(dm.sum(cost * x))
r = m.solve()
print("cost:", r.objective)
print("shipments:")
print(r.value(x))


## 2. Expression DAG

Every operator (`+`, `*`, `**`, `<=`, etc.) returns an Expression node, not a number. The Rust backend turns the DAG into the IR for the solver. Two practical implications:

- **Reuse** subexpressions; the DAG deduplicates them: `e = x[0]*x[1]; m.subject_to(e <= 10); m.minimize(e + x[2])`.
- **Avoid Python-level loops** for big models; prefer vectorized expressions or `sum()` over generators.

## 3. Common modeling tricks

| Want | How |
|------|-----|
| Either-or constraint | Big-M with binary indicator (Lesson 5) |
| Absolute value $|y - a|$ | Auxiliary $u \ge y - a$, $u \ge a - y$ (in min objective only) |
| Max of expressions $\max(e_1, e_2)$ | Aux $u \ge e_i$, minimize $u$ |
| Piecewise linear $f(x)$ | SOS2 or convex combination formulation |
| Logical $z = x \land y$ | $z \le x$, $z \le y$, $z \ge x + y - 1$ |

See {cite:p}`Williams2013,Wolsey1998` for the full catalogue.

## 4. Bounds matter

Tight bounds on variables are crucial:

- They **make presolve effective** {cite:p}`Savelsbergh1994,Belotti2009`.
- For nonconvex problems they **enable McCormick relaxations** {cite:p}`McCormick1976` (Lesson 16).
- Loose or missing bounds lead to long solves or unbounded LP relaxations.

If you don't know the right bound, *guess generously and shrink later*. `discopt` will warn if a variable is unbounded in a context where bounds are required.

## 5. Diagnosing a bad model

Before blaming the solver:

1. **Print `m.summary()`** — number of vars, constraints, nonzeros.
2. **Solve the LP relaxation** of an MILP — if it's infeasible, your model is wrong.
3. **Drop the objective, just check feasibility** — `m.solve()`.
4. **Slacken** suspect constraints with a penalty term — see which ones are tight in your "wrong" answer.

These tricks save days of debugging.

## References
{cite:p}`Williams2013` is the model-builder's bible; {cite:p}`Wolsey1998` for IP-specific tricks; {cite:p}`Belotti2009` for how presolve uses your bounds.